<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_post_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Review post processing

## Set up

In [ ]:
# @title Dependencies

# Installations
!pip install -q pandas tqdm

# First-party imports
from google.colab import drive
import os
import json

# Third-party imports
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 11.7 MB/s eta 0:00:00


In [ ]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews_processed")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews_judged")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_evaluated.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_evaluated.jsonl")

# Define the full log path (final Parquet output)
LOG_PATH = os.path.join(RAW_DIR, "llm_reviews_log.parquet")
# Define the full log path for intermediate JSONL storage
LOG_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_log.jsonl")

Mounted at /content/drive
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_processed.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged.


In [ ]:
# @title Setup definitions

# Error messages
JUDGE_GENERATION_FAILURE = "Error: Failed to generate judgement."
UNKNOWN_ERROR_STATE = "Error: Unknown client error."
EXTRACTION_ERROR = "Error: Failed to extract JSON from raw judgement"
PARSING_ERROR = "Error: Failed to parse judgement from JSON to single values"

EXPECTED_LABEL_KEYS = [
    'actionability_label',
    'grounding_specificity_label',
    'verifiability_label',
    'helpfulness_label'
]

In [ ]:
# @title Data import

# Read processed llm reviews
df_reviews = pd.read_parquet(os.path.join(DATASET_DIR, "llm_reviews_segmented.parquet"))

# Check df_reviews
display(df_reviews.shape)
display(df_reviews.head(3))


,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,extracted_weaknesses,extracted_scores,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating,segmented_comment,segment_hash
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",[Experimental details in the notes are summari...,"{'correctness_rating': 5.0, 'empirical_novelty...",5.0,4.0,4.0,Experimental details in the notes are summariz...,e3e2e8c694f2
1,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",[Experimental details in the notes are summari...,"{'correctness_rating': 5.0, 'empirical_novelty...",5.0,4.0,4.0,The optimal 2:4 MVUE is reported to be computa...,a9d37196da49
2,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",[Experimental details in the notes are summari...,"{'correctness_rating': 5.0, 'empirical_novelty...",5.0,4.0,4.0,Some practical engineering aspects are only di...,5e267a296029


(25, 16)

## Define

Original regex-based output checks from the authors `inference_utils.py` to enforce a clean final JSON-output structure: https://github.com/bodasadallah/RevUtil/blob/main/inference/inference_utils.py

In [ ]:
# @title inference_utils.py

import json
import re


known_keys = ['actionability_rationale', 'actionability_label', 'grounding_specificity_rationale', 'grounding_specificity_label', 'verifiability_rationale', 'verifiability_label', 'helpfulness_rationale', 'helpfulness_label']


def replace_category_names(text):
    category_mapping = {
        "1: Unverifiable": 1,
        "2: Borderline Verifiable": 2,
        "3: Somewhat Verifiable": 3,
        "4: Mostly Verifiable": 4,
        "5: Fully Verifiable": 5,
        "X: No Claim": "X",

        "1: Unactionable": 1,
        "2: Borderline Actionable": 2,
        "3: Somewhat Actionable": 3,
        "4: Mostly Actionable": 4,
        "5: Highly Actionable": 5,

        "1: Not Grounded": 1,
        "2: Weakly Grounded and Not Specific": 2,
        "3: Weakly Grounded and Specific": 3,
        "4: Fully Grounded and Under-Specific": 4,
        "5: Fully Grounded and Specific": 5,

        "1: Not Helpful at All": 1,
        "2: Barely Helpful": 2,
        "3: Somewhat Helpful": 3,
        "4: Mostly Helpful": 4,
        "5: Highly Helpful": 5
    }

    # Normalize dictionary for case-insensitive matching
    category_mapping_lower = {k.lower(): v for k, v in category_mapping.items()}
    partial_mapping_lower = {k.split(": ", 1)[-1].lower(): v for k, v in category_mapping.items()}

    def replace_match(match):
        matched_text = match.group(0).lower()  # Normalize case
        return str(category_mapping_lower.get(matched_text, partial_mapping_lower.get(matched_text, match.group(0))))

    # Replace full matches first (case-insensitive)
    pattern = re.compile(r'\b(?:' + '|'.join(map(re.escape, category_mapping_lower.keys())) + r')\b', re.IGNORECASE)
    text = pattern.sub(replace_match, text)

    # Replace partial matches (case-insensitive)
    pattern_partial = re.compile(r'\b(?:' + '|'.join(map(re.escape, partial_mapping_lower.keys())) + r')\b', re.IGNORECASE)
    text = pattern_partial.sub(replace_match, text)

    return text


expected_keys = [
    "actionability_rationale",
    "actionability_label",
    "grounding_specificity_rationale",
    "grounding_specificity_label",
    "verifiability_rationale",
    "verifiability_label",
    "helpfulness_rationale",
    "helpfulness_label"
]

import json
import re

import json

def extract_valid_json(text):
    label_keys = [
        "actionability_label",
        "grounding_specificity_label",
        "verifiability_label",
        "helpfulness_label"
    ]

    # Initialize result with None values
    result = {key: None for key in label_keys}

    # Match patterns like: "key": "1", "key": 1, or "key": "X"
    pattern = r'"(' + '|'.join(label_keys) + r')"\s*:\s*"?(X|\d+)"?'

    matches = re.findall(pattern, text)

    for key, val in matches:
        result[key] = str(val)  # Always store as string for valid JSON output

    return json.dumps(result, indent=2)



def escape_inner_quotes(text):
    """Finds specified rationale fields and escapes only internal double quotes."""
    fields = [
        "actionability_rationale",
        "grounding_specificity_rationale",
        "verifiability_rationale",
        "helpfulness_rationale"
    ]

    for field in fields:
        pattern = fr'("{field}"\s*:\s*")(.*?)("[\}},])'  # Escape closing brace
        matches = list(re.finditer(pattern, text, re.DOTALL))  # Find all matches first

        for match in reversed(matches):  # Process from last to first to avoid index shifting
            before, rationale, after = match.groups()
            escaped_rationale = rationale.replace('"', '\\"')  # Escape only inner quotes
            text = text[:match.start(2)] + escaped_rationale + text[match.end(2):]

    return text
def extract_dict(text):

    text = replace_category_names(text)  # Replace category names with numbers
    ## remove double spaces
    text = re.sub(' +', ' ', text)
    ## remove ``` from the text
    # text = text.replace('```', '')
    text = text.replace("-", "")  # Remove leading hyphens
    text = text.replace("\n", " ")  # Remove newlines
    text = text.replace("\\'", "'")  # Fix incorrectly escaped single quotes
    text = text.replace('\\"s', "'s")  # Fix incorrect escaped possessive 's
    text = text.replace("\\\\'", "\\\"")
    text = text.replace("\\\\", "\\")

    text = text.replace("[", "")  # Remove square brackets
    text = text.replace("]", "")  # Remove square brackets

    ############## For Prometheus2 #################
    # text = text.replace("[", '"')  # Replace single quotes with double quotes
    # text = text.replace("]", '"')  # Replace single quotes with double quotes

    ## if text begin with comma or space, remove it
    if text[0] == ',' or text[0] == ' ':
        text = text[1:]

    text = text.replace("\\\\", "\\") # Fix double backslashes
    dict_str  = ""
    if "```" in text:
        text = text + '#'
        match = re.search(r"```(?:json)?\s*(.*?)(```)?#", text, re.DOTALL)
        if match:
            text = match.group(0)
            ## remove the ```json  and ``` from the text
            text = text.replace('```json', '')
            text = text.replace('```', '')
            text = text.replace('#', '')

    text = text.strip()  # Remove leading and trailing whitespace

    if not text:
        return None

    ############# Comment for Prometheus2 ##############
    if text[0] != '{':
        text = '{' + text + '}'

    ################## cut the text if there is two newlines. This is for Prmetheus2 #########
    # if '\n\n' in text:
    #     halfs = text = text.split('\n\n', 1)
    #     text = halfs[0] if "actionability_label" in halfs[0] else halfs[1]

    if '{' not in text:
        text = '{' + text + '}'

    text = text.replace(" }  { ", ',')  # Remove newlines between dictionaries

    text = text.replace("\n", ' ')  # Remove newlines

    ############################# Prometheus2 ##########################
    # text = extract_valid_json(text)
    # text = text.replace('\\', '')  # Remove single quotes
    # print(f"Text after processing: {text} \n\n\n")

    ############ Some cases doesn't work with replacing the quotes, so trying both ways
    text2 = text
    try:
        text = text.replace("'", '"')  # Replace single quotes with double quotes
        text = escape_inner_quotes(text)  # Fix quotes inside rationale fields
        match = re.search(r'\{.*?\}', text, re.DOTALL)  # Extract first {...} block
        if match:
            dict_str = match.group()  # Get extracted dictionary string
        return json.loads(dict_str)  # Convert to Python dictionary safely
    except json.JSONDecodeError:
        print("Replacing quotes didn't work, trying without replacing quotes.")
        text = text2  # Revert to original text
        match = re.search(r'\{.*?\}', text, re.DOTALL)  # Extract first {...} block
        if match:
            dict_str = match.group()  # Get extracted dictionary string
        try:
            return json.loads(dict_str)  # Convert to Python dictionary safely
        except json.JSONDecodeError as e:
            print(f"Parsing error: {e}\nProblematic string: {dict_str}")
            return None

def extract_predictions(model_outputs):
    """
    Parses a list of model-generated texts to extract labels and returns a dictionary.

    :param model_outputs: List of strings containing model-generated text with labels.
    :return: List of dictionaries with extracted labels.
    """
    extracted_data = []

    for text in model_outputs:

        if 'outputs' in text.keys():
            text = text.outputs[0].text
        elif 'generated_text' in text.keys():
            text = text['generated_text']

        extracted_dict = extract_dict(text)
        if  not extracted_dict:
            extracted_data.append({
                'actionability_label': None,
                'grounding_specificity_label': None,
                'verifiability_label': None,
                'helpfulness_label': None,
                'actionability_rationale': None,
                'grounding_specificity_rationale': None,
                'verifiability_rationale': None,
                'helpfulness_rationale': None
            })
            continue

        parsed_result = {
            'actionability_label': str(extracted_dict.get('actionability_label', None)),
            'grounding_specificity_label':  str(extracted_dict.get('grounding_specificity_label', None)),
            'verifiability_label':  str(extracted_dict.get('verifiability_label', None)),
            'helpfulness_label':  str(extracted_dict.get('helpfulness_label', None)),
            ### rationale keys
            'actionability_rationale':  str(extracted_dict.get('actionability_rationale', None)),
            'grounding_specificity_rationale':  str(extracted_dict.get('grounding_specificity_rationale', None)),
            'verifiability_rationale':  str(extracted_dict.get('verifiability_rationale', None)),
            'helpfulness_rationale':  str(extracted_dict.get('helpfulness_rationale', None))
        }

        extracted_data.append(parsed_result)

    return extracted_data


In [ ]:
# @title Output parsing

def process_raw_llm_output(raw_output: str) -> dict:
    """
    Helper function to apply robust JSON extraction and cleaning to a raw output string.
    Returns a dictionary of the extracted data or an error dictionary.
    This function *always* returns a dictionary with the EXPECTED_LABEL_KEYS.
    """
    # Initialize a base result dictionary with None values for labels
    base_result = {key: None for key in EXPECTED_LABEL_KEYS}

    # 1. Handle explicit LLM generation failure string first (highest priority error)
    if raw_output == JUDGE_GENERATION_FAILURE:
        for key in EXPECTED_LABEL_KEYS:
            base_result[key] = JUDGE_GENERATION_FAILURE
        return base_result

    extracted_dict = None
    # 2. Phase 1: Attempt to extract the JSON dictionary from the raw string (Extraction Error)
    try:
        extracted_dict = extract_dict(raw_output)
    except Exception as e:
        print(f"Error (Extraction): Failed to extract JSON. {e}. Raw start: {raw_output[:100]}...")
        for key in EXPECTED_LABEL_KEYS:
            base_result[key] = EXTRACTION_ERROR
        return base_result

    # 3. Phase 2: If JSON extraction was successful, attempt to parse single values (Parsing Error)
    if extracted_dict:
        try:
            for key in EXPECTED_LABEL_KEYS:
                base_result[key] = str(extracted_dict.get(key, None))
            return base_result
        except Exception as e:
            print(f"Error (Parsing): Failed to parse single values. {e}. Raw start: {raw_output[:100]}...")
            for key in EXPECTED_LABEL_KEYS:
                base_result[key] = PARSING_ERROR
            return base_result
    else:
        # If extract_dict returned None, it means the extraction failed to yield a valid dictionary.
        # This falls under EXTRACTION_ERROR.
        print(f"Warning (Extraction): extract_dict returned None. Raw start: {raw_output[:100]}...")
        for key in EXPECTED_LABEL_KEYS:
            base_result[key] = EXTRACTION_ERROR
        return base_result

## Run

In [ ]:
# @title Execution

## Transformation

In [ ]:
# @title Conversion

# Convert RESULTS_JSON_PATH to RESULTS_PATH (Parquet)
if os.path.exists(RESULTS_JSON_PATH):
    try:
        print(f"Converting {RESULTS_JSON_PATH} to {RESULTS_PATH}...")
        jsonl_df = pd.read_json(RESULTS_JSON_PATH, lines=True)
        if not jsonl_df.empty:
            jsonl_df.to_parquet(RESULTS_PATH, index=False)
            print(f"Successfully converted and saved results to {RESULTS_PATH}.")
        else:
            print(f"{RESULTS_JSON_PATH} is empty. No Parquet file generated for results.")
    except Exception as e:
        print(f"Error converting results JSONL to Parquet: {e}")
else:
    print(f"Intermediate results file '{RESULTS_JSON_PATH}' does not exist. No results Parquet file generated.")

# Convert LOG_JSON_PATH to LOG_PATH (Parquet)
if os.path.exists(LOG_JSON_PATH):
    try:
        print(f"Converting {LOG_JSON_PATH} to {LOG_PATH}...")
        jsonl_log_df = pd.read_json(LOG_JSON_PATH, lines=True)
        if not jsonl_log_df.empty:
            jsonl_log_df.to_parquet(LOG_PATH, index=False)
            print(f"Successfully converted and saved log to {LOG_PATH}.")
        else:
            print(f"'{LOG_JSON_PATH}' is empty. No Parquet file generated for log.")
    except Exception as e:
        print(f"Error converting log JSONL to Parquet: {e}")
else:
    print(f"Intermediate log file '{LOG_JSON_PATH}' does not exist. No log Parquet file generated.")

In [ ]:
# @title Results

# Load result file
if os.path.exists(RESULTS_PATH):
    try:
        llm_results_df = pd.read_parquet(RESULTS_PATH)
        # Check results only if loading was successful
        display(llm_results_df)
        display(llm_results_df.shape)
    except Exception as e:
        print(f"Warning: Could not read existing results parquet file {RESULTS_PATH}. Error: {e}")
else:
    print(f"Results file '{RESULTS_PATH}' does not exist.")

# Load log file
if os.path.exists(LOG_PATH):
    try:
        llm_log_df = pd.read_parquet(LOG_PATH)
        # Check log only if loading was successful
        display(llm_log_df.head(5))
        display(llm_log_df.shape)
    except Exception as e:
        print(f"Warning: Could not read existing log parquet file {LOG_PATH}. Error: {e}")
else:
    print(f"Log file '{LOG_PATH}' does not exist.")